# NOTE: AF3 is only available on the chimera cluster. It will be removed prior to release

# AlphaFold3 Structure Prediction

This notebook demonstrates multi-modal structure prediction using AlphaFold3 from Google DeepMind. Unlike its predecessor, AlphaFold3 jointly predicts the structures of proteins, nucleic acids (DNA and RNA), ligands, and their complexes using a diffusion-based architecture that generates multiple structural samples per seed. We demonstrate `run_alphafold3` by predicting the structure of the MfnG protein bound to L-tyrosine, a practical example of protein-ligand complex prediction where both protein fold and small molecule geometry must be resolved simultaneously.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphafold3")
display_overview("alphafold3")
display_docs_section("alphafold3", "Background")

## Available tools

In [ ]:
display_available_tools("alphafold3")

### `run_alphafold3`

AlphaFold3 extends structure prediction beyond proteins to jointly model proteins, DNA, RNA, ligands, and their complexes in a single unified framework. Its diffusion-based architecture generates five structural samples per seed and automatically selects the highest-confidence prediction. SMILES strings for small molecules are automatically converted to CCD (Chemical Component Dictionary) codes internally. The `seeds` parameter controls stochastic sampling and can be extended to multiple values for more thorough exploration of conformational space, which is particularly useful for challenging tasks such as antibody-antigen docking.

In [ ]:
from pathlib import Path

from proto_tools import (
    AlphaFold3Config,
    AlphaFold3Input,
    Chain,
    StructurePredictionComplex,
    run_alphafold3,
)

In [ ]:
# Display input docs
display_api_reference("alphafold3", "input", "run_alphafold3")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = AlphaFold3Input(complexes=[complex])

In [ ]:
# Display config docs
display_api_reference("alphafold3", "config", "run_alphafold3")

# Configure AlphaFold3
config = AlphaFold3Config(
    name="mfng_ligand",
    seeds=[0],  # AlphaFold3 generates 5 diffusion samples per seed
    use_msa=False,
    verbose=False,
)

In [ ]:
# Run structure prediction
result = run_alphafold3(inputs, config)

In [ ]:
# Display output docs
display_api_reference("alphafold3", "output", "run_alphafold3")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Ranking score: {mfng_structure.ranking_score:.3f}")
print(f"  Average pLDDT: {mfng_structure.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.iptm:.3f}")

#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain. The protein backbone is shown alongside the bound L-tyrosine ligand, allowing you to inspect the predicted binding pose and overall fold quality.

In [ ]:
mfng_structure.visualize(style='stick', color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT scores for per-residue confidence visualization.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")